# Tiny LLM from Scratch

A character-level Transformer trained on Shakespeare.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)
print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [5]:
from google.colab import drive

drive.mount('/content/drive')


with open(
    "/content/drive/MyDrive/tiny_llm/input.txt",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
chars = sorted(list(set(text)))
print("Vocabulary:", chars)
print("Vocabulary size:", len(chars))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encoded = [stoi[ch] for ch in text]
data = torch.tensor(encoded, dtype=torch.long)


Vocabulary: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary size: 65


In [7]:
split_index = int(0.9 * len(data))

train_data = data[:split_index]
val_data = data[split_index:]

print("Total characters:", len(text))
print("Vocabulary size:", vocab_size)
print("Train size:", len(train_data))
print("Validation size:", len(val_data))


Total characters: 1115394
Vocabulary size: 65
Train size: 1003854
Validation size: 111540


In [8]:
block_size = 128

batch_size = 32

n_embd = 128

num_heads = 4

learning_rate = 0.001

num_steps = 5001

dropout_rate = 0.2

## Attention


In [9]:
class Head(nn.Module):

    def __init__(self, head_size, n_embd):
        super().__init__()

        self.head_size = head_size

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.dropout = nn.Dropout(dropout_rate)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        attention_score = (
            q @ k.transpose(-2, -1)
        ) / (self.head_size ** 0.5)

        attention_score = attention_score.masked_fill(
            self.causal_mask[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(
            attention_score,
            dim=-1
        )

        attention_weights = self.dropout(attention_weights)

        output = attention_weights @ v

        return output


In [10]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size, n_embd)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):

        outputs = [
            head(x)
            for head in self.heads
        ]

        out = torch.cat(outputs, dim=-1)

        out = self.proj(out)
        out = self.dropout(out)

        return out


## Feed Forward Network


In [11]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.net(x)


## Transformer Block


In [12]:
class Block(nn.Module):

    def __init__(self, n_embd, num_heads):
        super().__init__()

        head_size = n_embd // num_heads

        self.attention = MultiHeadAttention(
            num_heads,
            head_size,
            n_embd
        )

        self.feed_forward = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.attention(self.ln1(x))

        x = x + self.feed_forward(self.ln2(x))

        return x


## TinyLLM


In [13]:
class TinyLLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        n_embd,
        num_heads,
        num_layers=4
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding = nn.Embedding(
            block_size,
            n_embd
        )

        self.transformer = nn.Sequential(*[
            Block(n_embd, num_heads)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, x):

        B, T = x.shape

        token_emb = self.token_embedding(x)

        pos_id = torch.arange(
            T,
            device=x.device
        )

        pos_emb = self.position_embedding(pos_id)

        x = token_emb + pos_emb

        x = self.transformer(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits


## Create Model


In [14]:
model = TinyLLM(
    vocab_size,
    n_embd,
    num_heads
).to(device)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters()))

print("\nParameters by layer:\n")

for name, param in model.named_parameters():
    print(f"{name:60} {param.numel():>8}")


Number of parameters:
824897

Parameters by layer:

token_embedding.weight                                           8320
position_embedding.weight                                       16384
transformer.0.attention.heads.0.key.weight                       4096
transformer.0.attention.heads.0.query.weight                     4096
transformer.0.attention.heads.0.value.weight                     4096
transformer.0.attention.heads.1.key.weight                       4096
transformer.0.attention.heads.1.query.weight                     4096
transformer.0.attention.heads.1.value.weight                     4096
transformer.0.attention.heads.2.key.weight                       4096
transformer.0.attention.heads.2.query.weight                     4096
transformer.0.attention.heads.2.value.weight                     4096
transformer.0.attention.heads.3.key.weight                       4096
transformer.0.attention.heads.3.query.weight                     4096
transformer.0.attention.heads.3.value.

## Batch and Validation Functions


In [15]:
def get_batch(data_source):

    starts = torch.randint(
        0,
        len(data_source) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        data_source[start:start + block_size]
        for start in starts
    ]).to(device)

    y = torch.stack([
        data_source[start + 1:start + block_size + 1]
        for start in starts
    ]).to(device)

    return x, y


def estimate_loss():

    losses = {}

    for split, data_source in [
        ("train", train_data),
        ("val", val_data)
    ]:

        model.eval()

        split_losses = []

        with torch.no_grad():

            for _ in range(20):

                x, y = get_batch(data_source)

                logits = model(x)

                B, T = x.shape

                logits = logits.reshape(
                    B * T,
                    vocab_size
                )

                targets = y.reshape(B * T)

                loss = F.cross_entropy(
                    logits,
                    targets
                )

                split_losses.append(loss.item())

        losses[split] = sum(split_losses) / len(split_losses)

    model.train()

    return losses


## Train Model

Run this once, then save the model.


In [20]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [ ]:
model.train()

for i in range(num_steps):

    x, y = get_batch(train_data)

    optimizer.zero_grad()

    logits = model(x)

    B, T = x.shape

    logits = logits.reshape(
        B * T,
        vocab_size
    )

    targets = y.reshape(B * T)

    loss = F.cross_entropy(
        logits,
        targets
    )

    loss.backward()

    optimizer.step()

    if i % 500 == 0:

        losses = estimate_loss()

        print(
            f"Step {i} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f}"
        )

print("\nTraining finished!")


Step 0 | Train Loss: 3.9589 | Val Loss: 3.9742
Step 500 | Train Loss: 2.0326 | Val Loss: 2.0801
Step 1000 | Train Loss: 1.7287 | Val Loss: 1.8603
Step 1500 | Train Loss: 1.5982 | Val Loss: 1.7825
Step 2000 | Train Loss: 1.5236 | Val Loss: 1.7191
Step 2500 | Train Loss: 1.4729 | Val Loss: 1.6781
Step 3000 | Train Loss: 1.4689 | Val Loss: 1.6403
Step 3500 | Train Loss: 1.4214 | Val Loss: 1.6283
Step 4000 | Train Loss: 1.4214 | Val Loss: 1.6095
Step 4500 | Train Loss: 1.3935 | Val Loss: 1.5766
Step 5000 | Train Loss: 1.3749 | Val Loss: 1.5941

Training finished!


## Save Trained Model


In [21]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "stoi": stoi,
        "itos": itos,
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "num_heads": num_heads,
    },
    "tiny_shakespeare_model.pth"
)

print("Model saved successfully!")

Model saved successfully!


## Load Trained Model Later

After restarting Jupyter, run the setup/model cells, then run this instead of training.


In [16]:
model = TinyLLM(
    vocab_size,
    n_embd,
    num_heads
)

model.load_state_dict(
    torch.load(
        "tiny_llm_shakespeare.pth",
        map_location=device
    )
)

model.to(device)

model.eval()

print("Trained model loaded!")


Trained model loaded!


## Generate Text


In [17]:
def generate(
    start_text,
    max_tokens=500,
    temperature=0.3,
    top_k=10
):

    model.eval()

    context = torch.tensor(
        [[stoi[ch] for ch in start_text]],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        for step in range(max_tokens):

            context_for_model = context[:, -block_size:]

            logits = model(context_for_model)

            logits = logits[:, -1, :]

            # Check logits
            if torch.isnan(logits).any():
                print("❌ NaN found in logits at step:", step)
                break

            if torch.isinf(logits).any():
                print("❌ Inf found in logits at step:", step)
                break

            logits = logits / temperature

            # Top-k filtering
            if top_k is not None:

                values, _ = torch.topk(
                    logits,
                    top_k
                )

                cutoff = values[:, -1].unsqueeze(-1)

                logits = torch.where(
                    logits < cutoff,
                    torch.tensor(
                        float("-inf"),
                        device=device
                    ),
                    logits
                )

            probs = F.softmax(
                logits,
                dim=-1
            )

            # SUPER IMPORTANT CHECK
            if torch.isnan(probs).any():
                print("❌ NaN found in probabilities at step:", step)
                print("Logits:", logits)
                break

            if torch.isinf(probs).any():
                print("❌ Inf found in probabilities at step:", step)
                break

            print(
                f"Step {step} | "
                f"min={probs.min().item():.6f} | "
                f"max={probs.max().item():.6f} | "
                f"sum={probs.sum().item():.6f}"
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            context = torch.cat(
                (context, next_token),
                dim=1
            )

    generated_text = ''.join(
        itos[token.item()]
        for token in context[0]
    )

    return generated_text

In [ ]:
print(generate(
    start_text="ROMEO:",
    max_tokens=500,
    temperature=0.3,
     top_k=10
))
